In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# API Proxy Client Usage

This notebook demonstrates how to interact with the API Management gateway using the `google-genai` 2.x client library.

## Install Dependencies

This cell installs the `google-genai` 2.x library, which is required to interact with Google's Gemini and Gemini Omni models.

In [ ]:
%pip install 'google-genai>=2.34.0,<3.0.0'

## Set Variables

In [ ]:
import subprocess

GCP_PROJECT_ID = subprocess.run(['terraform', 'output', '-raw', 'gemini_project_id'], capture_output=True, text=True, check=True).stdout.strip()
GCP_LOCATION = "global"
GATEWAY_URL = subprocess.run(['terraform', 'output', '-raw', 'api_management_gateway_url'], capture_output=True, text=True, check=True).stdout.strip()
API_KEY = subprocess.run(['terraform', 'output', '-raw', 'api_management_subscription_key'], capture_output=True, text=True, check=True).stdout.strip()

# Initialize Client

In [ ]:
import base64
from google import genai
from google.genai import types
from IPython.display import display, Image, Markdown, Video

client = genai.Client(
    enterprise=True,
    http_options=types.HttpOptions(
        base_url=f"{GATEWAY_URL}/vertex-ai",
        base_url_resource_scope=types.ResourceScope.COLLECTION,
        headers={
            "Ocp-Apim-Subscription-Key": API_KEY
        },
        client_args={"verify": True},
    )
)

# Generate Content

In [ ]:
response = client.models.generate_content(
    model="gemini-4-argon",
    contents="Tell me a joke about space."
)

print(response.text)

# Streaming Generate Content

In [ ]:
for chunk in client.models.generate_content_stream(
    model="gemini-3.8-flash", contents="Tell me a story in 300 words."
):
    print(chunk.text, end='')

# Nano Banana Image Generation

In [ ]:
response = client.models.generate_content(
    model="gemini-3.1-flash-image",
    contents=("Generate an image of the Eiffel tower with fireworks in the background."),
    config=types.GenerateContentConfig(
        response_modalities=[types.Modality.TEXT, types.Modality.IMAGE],
        candidate_count=1,
    ),
)
for part in response.candidates[0].content.parts:
    if part.text:
        display(Markdown(part.text))
    elif part.inline_data:
        display(Image(data=part.inline_data.data, width=500))

# Gemini Omni Video Generation

Generate video with synchronized audio using `gemini-omni-1.2-flash` via the Interactions API.

In [ ]:
interaction = client.interactions.create(
    model="gemini-omni-1.2-flash",
    input="Generate a video of a close-up shot of a golden retriever playing in a field of sunflowers, with sunlight filtering through the petals.",
    store=False,
)

with open("video.mp4", "wb") as out_file:
    out_file.write(base64.b64decode(interaction.output_video.data))

display(Video("video.mp4", embed=True, width=600, height=400))